# Detection baselines: LettuceDetect, HHEM, Granite Guardian

Scores the released detectors on the same development items as Channel A: AUROC, false alarms on clean answers, wrong-context recall. Granite Guardian (cut-list item 2) runs at 4-bit on 1,000 items. Expect roughly 2-4 GPU hours.

**On Kaggle:** Create → New Notebook → File → Import Notebook → this file. Settings: *Accelerator* **GPU T4 x2**, *Internet* **on**.
For long jobs use **Save Version → Save & Run All (Commit)**: it runs in the background for up to 12 hours and keeps everything in `/kaggle/working/outputs`.
To continue in a later notebook, add this notebook's output as input (*Add Input → Your Work*).

**Afterwards:** download `outputs/*/results.json` and `report.md` files into `runs/` in the repository, and log the session's GPU hours in `logs/gpu-hours.csv`. Checkpoints are saved regularly; re-running the same cell resumes from the latest one.

In [ ]:
# Setup: clone FaithGuard, install it with the pinned GPU packages, fetch the data.
import os, subprocess, sys
WORK = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
os.chdir(WORK)
if not os.path.exists('faithguard'):
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/SharafThawfeek/FaithGurd-new.git', 'faithguard'])
os.chdir('faithguard')
REQUIREMENTS = ['transformers==5.19.0', 'peft==0.21.2', 'accelerate==1.15.0', 'bitsandbytes==0.50.2', 'huggingface_hub==1.33.0', 'datasets==5.1.0', 'lettucedetect==0.2.3']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
OUT = os.path.join(WORK, 'outputs'); os.makedirs(OUT, exist_ok=True)
!faithguard data download all
!nvidia-smi --query-gpu=name,memory.total --format=csv

## LettuceDetect v1-large and v2 mmBERT

In [ ]:
!python -m faithguard.train.baseline_eval --detector lettuce-v1-large --out {OUT}/base-lettuce-v1
!python -m faithguard.train.baseline_eval --detector lettuce-v2-mmbert --out {OUT}/base-lettuce-v2

## HHEM-2.1-Open

In [ ]:
!python -m faithguard.train.baseline_eval --detector hhem --out {OUT}/base-hhem

## Granite Guardian 4.1-8B (4-bit)

In [ ]:
!python -m faithguard.train.baseline_eval --detector granite --limit 1000 --out {OUT}/base-granite

## Results

In [ ]:
import glob, json
for path in sorted(glob.glob(OUT + '/**/results.json', recursive=True)):
    print(path)
    print(json.dumps({k: v for k, v in json.load(open(path)).items() if k in ('summary', 'channel_a', 'channel_b', 'auroc', 'false_alarm_clean', 'wrong_context_recall')}, indent=1)[:1500])